# evaluate — Direction 1: a policy in the organisers' simulator, against the reference runs

Runs a policy for 30 days (`hvac_controller/d1/evaluate.py`), one fresh process per scenario × seed, and compares
it per scenario with the reference runs in `buildings` (K10 and V1).

- `POLICY = "pkg"`: the attached `hvac-submission` package, run with its own `d1_policy.py`; its results are named
  `TAG`. `POLICY = "k10"`: the public baseline.
- `SCENARIOS = "held_out"`: the 19 held-out scenarios (building_train_2's 5 shipped strips, building_train_2 × 2
  new stations, 6 held-out generated buildings × 2 strips); `"shipped"`: the 30 shipped ones.
- `WHERE`: `original` (the truth) and / or `replica` (the NumPy replica, from the caches).
- `SEEDS`: seed 0 sees its observations as they are; seeds ≥ 1 add 1e-6 noise to the state observations (the
  closed loop is chaotic, D1 doc §2.7).
- The jobs are sorted longest first (times from the reference runs) and dealt to `N_SHARDS` sessions. The large
  held-out buildings take ≈ 2–2.3 h each, so 3 sessions finish the 19 in ≈ 2.3 h.

**Inputs:** the competition data, `buildings`, and `hvac-submission` for `POLICY = "pkg"`. CPU, Internet on.

**Output:** `/kaggle/working/score_<TAG>_<run>.zip`: one JSON per run; the comparison is printed.

In [ ]:
REPO_URL = "https://github.com/CryAndRRich/hvac-controller"
REPO_REF = "direction1"
POLICY, TAG = "pkg", "v2"  # "pkg" (hvac-submission, named TAG) or "k10"
SCENARIOS = "held_out"     # "held_out" (19) or "shipped" (30)
WHERE, SEEDS = "original", 1
RUN_ID, N_SHARDS = 0, 3
N_PROCS = 4                # simulator runs at once

In [ ]:
import json, os, shutil, subprocess, sys, zipfile
from pathlib import Path

REPO, INPUT, WORK = Path("/tmp/hvac-controller"), Path("/kaggle/input"), Path("/tmp/work")
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "--quiet", REPO_URL, str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--quiet", REPO_REF], check=True)
print(subprocess.run(["git", "-C", str(REPO), "log", "-1", "--oneline"], capture_output=True, text=True).stdout)
sys.path.insert(0, str(REPO))
ENV = dict(os.environ, PYTHONPATH=str(REPO))


def dataset(slug):
    """One of the four datasets (docs/KAGGLE.md), wherever Kaggle mounts it."""
    return next(p for p in (INPUT / "datasets" / "cryandrrich" / slug, INPUT / slug) if p.is_dir())


SHIPPED = next(p for p in sorted(INPUT.rglob("training_scenarios"))
               if (p / "building_train_1" / "atlanta_2" / "config.gin").exists())
ROOT = REPO / "smart_buildings" / "training_scenarios"
shutil.copytree(SHIPPED, ROOT, dirs_exist_ok=True)
DATA = dataset("buildings")
if POLICY == "pkg":
    PKG = WORK / TAG  # the package, run with its own code; the folder names its results
    shutil.copytree(dataset("hvac-submission"), PKG, dirs_exist_ok=True)
    print((PKG / "README.txt").read_text())
    POLICIES = f"pkg:{PKG}"
else:
    POLICIES = TAG = "k10"

shipped = sorted(f"{p.parent.parent.name}/{p.parent.name}" for p in ROOT.glob("*/*/config.gin"))
if SCENARIOS == "held_out":  # the generated ones are rebuilt from the dataset's YAMLs and weather presets
    for p in DATA.glob("building_0*.yaml"):
        shutil.copy(p, REPO / "smart_buildings" / "buildings" / p.name)
    (REPO / "smart_buildings" / "weather" / "csv").mkdir(parents=True, exist_ok=True)
    for p in DATA.glob("x*.yaml"):
        shutil.copy(p, REPO / "smart_buildings" / "weather" / p.name)
    for p in DATA.glob("x*.csv"):
        shutil.copy(p, REPO / "smart_buildings" / "weather" / "csv" / p.name)
    tags = ("gen_val", "shipped_building_new_station")
    r = subprocess.run([sys.executable, "-m", "hvac_controller.d1.augment", "scenarios", str(DATA / "manifest_v2.json"),
                        "--tags", ",".join(tags)], cwd=REPO, env=ENV, capture_output=True, text=True)
    print("held-out scenario folders: exit", r.returncode, (r.stdout + r.stderr)[-1500:])
    assert r.returncode == 0
    manifest = json.loads((DATA / "manifest_v2.json").read_text())
    held_out = {e["building"] for e in manifest if e["tag"] == "shipped_building_new_station"}  # building_train_2
    LIST = [s for s in shipped if s.split("/")[0] in held_out] + [e["label"].replace("__", "/", 1) for e in manifest
                                                                  if e["tag"] in tags]
else:
    LIST = shipped
REFS = [json.loads(p.read_text()) for p in sorted(DATA.glob("original__*.json")) + sorted(DATA.glob("replica__*.json"))]
seconds = {}  # of each scenario's 30 days in the original simulator, from the reference runs
for r in REFS:
    if r["where"] == "original":
        seconds[r["scenario"]] = max(seconds.get(r["scenario"], 0.0), r["seconds"])
HERE = sorted(LIST, key=lambda s: seconds.get(s, 0.0), reverse=True)[RUN_ID::N_SHARDS]
print(f"{len(HERE)} of {len(LIST)} scenarios here, the longest {max(seconds.get(s, 0.0) for s in HERE) / 3600:.1f} h:", HERE)

In [ ]:
# The runs, N_PROCS at a time; each result is printed as it ends.
EVAL = WORK / "eval"
cmd = [sys.executable, "-m", "hvac_controller.d1.evaluate", str(EVAL), "--scenarios", ",".join(HERE),
       "--scenario-root", str(ROOT), "--cache-dir", str(DATA), "--policies", POLICIES, "--seeds", str(SEEDS),
       "--where", WHERE, "--n-procs", str(N_PROCS)]
proc = subprocess.Popen(cmd, cwd=REPO, env=ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end="", flush=True)
print("exit", proc.wait())

In [ ]:
# Against the reference runs, the gate and the zip.
from hvac_controller.d1 import evaluate as ev

results = [json.loads(p.read_text()) for p in sorted(EVAL.glob("*.json"))]
for r in results:
    if "error" in r:
        print("ERROR", r["where"], r["scenario"], r["seed"], r["error"], r["traceback"][-1500:])
ens = ev.ensembles(results + [r for r in REFS if ev.policy_tag(r["policy"]) != TAG])
for where in WHERE.split(","):
    for ref in ("k10", "v1"):
        cmp = ev.compare(ens, TAG, ref, where)
        if not cmp:
            continue
        print(f"\n{TAG} vs {ref} ({where}):")
        for scenario, c in sorted(cmp.items()):
            if scenario != "mean":
                print(f"  {scenario:34s} {c['candidate']:.5f} vs {c['null']:.5f} ({c['rel']:+.1%}, z {c['z']:+.1f})")
        m = cmp["mean"]
        print(f"  mean over {len(cmp) - 1}: {m['candidate']:.5f} vs {m['null']:.5f} ({m['candidate'] / m['null'] - 1:+.1%})")
want = len(HERE) * len(WHERE.split(",")) * SEEDS
print(f"\nGATE: {len(results)} runs (want {want}), no errors:",
      len(results) == want and not any("error" in r for r in results))
name = Path(f"/kaggle/working/score_{TAG}_{RUN_ID}.zip")
with zipfile.ZipFile(name, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(EVAL.glob("*.json")):
        z.write(p, p.name)
print("wrote", name)